# Manual vanilla RNN Seq2Seq

A source token ID becomes an embedding $x^{\langle t\rangle}$ with shape `[B, D]`. With column-vector weight names $W_{hx}\in\mathbb{R}^{H\times D}$ and $W_{hh}\in\mathbb{R}^{H\times H}$, the batch-row computation is

$$z_h^{\langle t\rangle}=x^{\langle t\rangle}W_{hx}^{\mathsf T}+h^{\langle t-1\rangle}W_{hh}^{\mathsf T}+b_h,\qquad h^{\langle t\rangle}=\tanh(z_h^{\langle t\rangle}).$$

The encoder carries the state only through real source tokens. Its final state initializes a decoder with a separate embedding and separate RNN weights. The decoder computes $z_y^{\langle t\rangle}=h^{\langle t\rangle}W_{yh}^{\mathsf T}+b_y$ with shape `[B, V_vi]`. Logits stay unnormalized for cross-entropy.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'experiments' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
import torch
from src.model import ModelConfig, Seq2Seq, VanillaRNNCell
cell = VanillaRNNCell(3, 5)
x, h = torch.randn(2, 3), torch.randn(2, 5)
assert torch.allclose(cell(x, h), torch.tanh(x @ cell.W_hx.T + h @ cell.W_hh.T + cell.b_h))
print('One RNN step:', tuple(cell(x, h).shape))

In [ ]:
model = Seq2Seq(ModelConfig(12, 15, embedding_dim=4, hidden_dim=5))
source = torch.tensor([[1, 4, 2, 0], [1, 5, 6, 2]])
lengths = torch.tensor([3, 4])
decoder_input = torch.tensor([[1, 7, 8], [1, 9, 0]])
logits = model(source, lengths, decoder_input)
assert logits.shape == (2, 3, 15)
assert torch.allclose(model.encoder(source, lengths), model.encoder(torch.nn.functional.pad(source, (0, 2)), lengths))
print('Decoder logits:', tuple(logits.shape))

In [5]:
import torch 

batch_size = 2
embed_dim = 3
hidden_dim = 4

x_t = torch.randn(batch_size, embed_dim)
h_prev = torch.zeros(batch_size, hidden_dim)
W_hx = torch.randn(hidden_dim, embed_dim, requires_grad=True)
W_hh = torch.randn(hidden_dim, hidden_dim, requires_grad=True)
b_h = torch.zeros(hidden_dim, requires_grad=True)

z_h = x_t @ W_hx.T + h_prev @ W_hh.T + b_h
h_t = torch.tanh(z_h)

print("x_t:", x_t.shape)
print("h_prev:", h_prev.shape)
print("W_hx:", W_hx.shape)
print("W_hh:", W_hh.shape)
print("z_h:", z_h.shape)
print("h_t:", h_t.shape)

h_t.sum().backward()
print("W_hx gradient:", W_hx.grad.shape)
print("W_hh gradient:", W_hh.grad.shape)

x_t: torch.Size([2, 3])
h_prev: torch.Size([2, 4])
W_hx: torch.Size([4, 3])
W_hh: torch.Size([4, 4])
z_h: torch.Size([2, 4])
h_t: torch.Size([2, 4])
W_hx gradient: torch.Size([4, 3])
W_hh gradient: torch.Size([4, 4])
